### **Naïve Bayes Implementation**

***Performing it manually***

In [10]:
import re
import math
from collections import defaultdict, Counter

dataset = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your  meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"), 
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

In [15]:
def tokenize(text):
    text = text.lower()
    words = re.findall(r"[a-z0-9]+(?:'[a-z]+)*", text)
    return words

#### **a. Generate a Bag of Words (for word frequency)**

In [85]:
def build_vocabulary(dataset):
    vocab = set()
    word_counts = defaultdict(Counter)


    for text, label in dataset:
        words = tokenize(text)
        vocab.update(words)
        word_counts[label].update(words)
    
    return sorted(vocab), word_counts

vocabulary, word_counts = build_vocabulary(dataset)

print("\nVocabulary Size:", len(vocabulary))

# Sort vocabulary alphabetically for clean table
sorted_vocab = sorted(vocabulary)


print("\n  Word Frequency")

for word in sorted_vocab:
    frequency_count = sum(word_counts[label][word] for label in word_counts.keys())
    print("{:<10} {:<5} {:<10}".format(word, "=", frequency_count))


Vocabulary Size: 44

  Word Frequency
3          =     1         
50         =     1         
a          =     1         
are        =     2         
at         =     2         
can        =     1         
catch      =     1         
click      =     1         
dinner     =     1         
for        =     3         
free       =     2         
get        =     1         
here       =     1         
hi         =     1         
how        =     1         
in         =     1         
iphone     =     1         
let's      =     1         
limited    =     1         
lowest     =     1         
meds       =     1         
meeting    =     2         
mom        =     1         
money      =     1         
now        =     1         
off        =     1         
office     =     2         
on         =     1         
pm         =     1         
price      =     1         
prizes     =     1         
report     =     1         
send       =     1         
still      =     1         
team     

#### **b. Calculate the prior for the class** ___HAM___ and ___SPAM___

In [86]:
def calculate_priors(dataset):
    total_docs = len(dataset)
    class_counts = Counter(label for _, label in dataset)
    
    priors = {
        cls: class_counts[cls] / total_docs
        for cls in class_counts
    }
    
    return priors

priors = calculate_priors(dataset)

print("\nTotal Documents:", len(dataset))
print("Class Counts:", dict(Counter(label for _, label in dataset)))

print("\nClass Priors")
for cls, prior in priors.items():
    print("{:<5} {:<2} {:<10}".format(cls, "=", prior))



Total Documents: 11
Class Counts: {'SPAM': 5, 'HAM': 6}

Class Priors
SPAM  =  0.45454545454545453
HAM   =  0.5454545454545454


#### **c. Calculate the likelihood of the tokens in the vocabulary with respect to the class.**

In [100]:
def calculate_likelihoods(vocabulary, word_counts):
    likelihoods = {cls: {} for cls in word_counts.keys()}
    vocab_size = len(vocabulary)
    
    for cls in word_counts.keys():
        total_words_in_class = sum(word_counts[cls].values())
        for word in vocabulary:
            word_freq = word_counts[cls].get(word, 0)
            likelihoods[cls][word] = (word_freq + 1) / (total_words_in_class + vocab_size)
    return likelihoods

# Build everything
vocabulary, word_counts = build_vocabulary(dataset)
priors = calculate_priors(dataset)
likelihoods = calculate_likelihoods(vocabulary, word_counts)

print("\nLikelihoods:")
print("\n{:<17} {:<9} {:<9}".format("Word", "HAM", "SPAM"))
print("-" * 35)

for word in vocabulary:
    ham_likelihood = likelihoods['HAM'][word]
    spam_likelihood = likelihoods['SPAM'][word]
    print("{:<15} {:<10.5f} {:<10.5f}".format(word, ham_likelihood, spam_likelihood))


Likelihoods:

Word              HAM       SPAM     
-----------------------------------
3               0.02597    0.01515   
50              0.01299    0.03030   
a               0.01299    0.03030   
are             0.03896    0.01515   
at              0.03896    0.01515   
can             0.02597    0.01515   
catch           0.02597    0.01515   
click           0.01299    0.03030   
dinner          0.02597    0.01515   
for             0.02597    0.04545   
free            0.01299    0.04545   
get             0.01299    0.03030   
here            0.01299    0.03030   
hi              0.02597    0.01515   
how             0.02597    0.01515   
in              0.02597    0.01515   
iphone          0.01299    0.03030   
let's           0.02597    0.01515   
limited         0.01299    0.03030   
lowest          0.01299    0.03030   
meds            0.01299    0.03030   
meeting         0.03896    0.01515   
mom             0.02597    0.01515   
money           0.01299    0.03030   

#### **d. Determine the class of the following test sentence:**

*i. Limited offer, click here!*

*ii. Meeting at 2 PM with the manager.*

In [101]:
def classify(sentence, priors, likelihoods, vocabulary, word_counts):
    words = tokenize(sentence)
    
    scores = {}
    
    for cls in ["HAM", "SPAM"]:
        # Start with log prior
        score = math.log(priors[cls])
        
        total_words_in_class = sum(word_counts[cls].values())
        vocab_size = len(vocabulary)
        
        for word in words:
            if word in vocabulary:
                score += math.log(likelihoods[cls][word])
            else:
                # Handle unknown words with Laplace smoothing
                score += math.log(1 / (total_words_in_class + vocab_size))
        
        scores[cls] = score
    
    return max(scores, key=scores.get), scores


In [102]:
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

print("\nManual Naïve Bayes Classification:\n")

for sentence in test_sentences:
    predicted_class, scores = classify(sentence, priors, likelihoods, vocabulary, word_counts)
    print(f"Sentence: {sentence}")
    print(f"Predicted Class: {predicted_class}")
    print(f"Scores: {scores}\n")


Manual Naïve Bayes Classification:

Sentence: Limited offer, click here!
Predicted Class: SPAM
Scores: {'HAM': -17.98135749098505, 'SPAM': -15.467634786790136}

Sentence: Meeting at 2 PM with the manager.
Predicted Class: HAM
Scores: {'HAM': -26.73610763753005, 'SPAM': -30.11604055454925}



#### **Using Scikit-Learn.**

#### **a. Determine the class of the following test sentence:**
*i. Limited offer, click here!*

*ii. Meeting at 2 PM with the manager.*

In [103]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Prepare data for scikit-learn

texts = [text for text, label in dataset]
labels = [label for text, label in dataset]

# Convert text to Bag of Words
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

# Train Multinomial Naïve Bayes
model = MultinomialNB()
model.fit(X, labels)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [108]:
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

X_test = vectorizer.transform(test_sentences)
predictions = model.predict(X_test)
proba = model.predict_proba(X_test)

print("\nScikit-Learn MultinomialNB Classification:")

for sentence, prediction, probs in zip(test_sentences, predictions, proba):
    print(f"\nSentence: {sentence}")
    print(f"Predicted Class: {prediction}")
    # Print probabilities for each class
    for cls, p in zip(model.classes_, probs):
        print(f"  P({cls}) = {p:.5f}")


Scikit-Learn MultinomialNB Classification:

Sentence: Limited offer, click here!
Predicted Class: SPAM
  P(HAM) = 0.08472
  P(SPAM) = 0.91528

Sentence: Meeting at 2 PM with the manager.
Predicted Class: HAM
  P(HAM) = 0.97844
  P(SPAM) = 0.02156
